# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment using Python 3.12 explicitly
# !~/.local/bin/uv venv .venv --seed --python 3.12 --clear

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"
# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(usually named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment.

In [ ]:
# # activate venv after installation. This needs to be run everytime.
# !source ./.venv/bin/activate
# print("done")

done


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [1]:
# !pip uninstall -y vllm vllm-flash-attn flashinfer-python flashinfer-cubin humming-kernels tokenspeed-mla tokenspeed-triton quack-kernels

# !pip install -U pip

# !pip install "transformers<=4.57" sympy numpy tqdm bitsandbytes "antlr4-python3-runtime==4.11.1"

# !pip install "vllm==0.11.1" --extra-index-url https://download.pytorch.org/whl/cu128

Found existing installation: vllm 0.11.1
Uninstalling vllm-0.11.1:
  Successfully uninstalled vllm-0.11.1
Found existing installation: flashinfer-python 0.5.2
Uninstalling flashinfer-python-0.5.2:
  Successfully uninstalled flashinfer-python-0.5.2
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
  Using cached vllm-0.11.1-cp38-abi3-manylinux1_x86_64.whl.metadata (18 kB)
  Using cached flashinfer_python-0.5.2-py3-none-any.whl.metadata (11 kB)
Using cached vllm-0.11.1-cp38-abi3-manylinux1_x86_64.whl (370.7 MB)
Using cached flashinfer_python-0.5.2-py3-none-any.whl (6.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [vllm]


In [2]:
%cd /content/151B_SP26_Competition

import torch
import transformers
from vllm import LLM, SamplingParams
import vllm

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__)
print("vllm:", vllm.__version__)
print("vLLM LLM import works")

/content/151B_SP26_Competition
torch: 2.9.0+cu128
cuda: 12.8
gpu: NVIDIA A100-SXM4-40GB
transformers: 4.56.2
vllm: 0.11.1
vLLM LLM import works


In [3]:
import json
import os
import re
import sys
import gc
from pathlib import Path
from typing import Optional

import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"
DATA_PATH   = "data/private.jsonl"
OUTPUT_PATH = "results/verification-pe-colab-16k.jsonl"

MAX_TOKENS = 16384

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

Path("results").mkdir(exist_ok=True)

print("done")
print("MAX_TOKENS:", MAX_TOKENS)
print("OUTPUT_PATH:", OUTPUT_PATH)

done
MAX_TOKENS: 16384
OUTPUT_PATH: results/verification-pe-colab-16k.jsonl


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)

print(f"Loaded {len(data)} questions ({n_mcq} MCQ, {n_free} free-form)")

mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))

print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 943 questions (300 MCQ, 643 free-form)

── MCQ sample ──
{
  "question": "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().",
  "options": [
    "Unchanged",
    "Increased by ten percent",
    "Reduced by one percent",
    "Increased by one percent",
    "Decreased by ten percent",
    "Halved",
    "Unable to determine",
    "Doubled",
    "Decreased by five percent",
    "Expanded tenfold"
  ],
  "id": 1
}

── Free-form sample ──
{
  "question": "Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]\nb) $4 \\cdot 3-2+2 \\cdot 3=$ [ANS]",
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
SYSTEM_PROMPT_MATH = (
    "You are a precise mathematical reasoner. Solve the following problem rigorously."
    "After obtaining an answer, independently check it for errors or contradictions. "
    "Return only the corrected final solution inside \\boxed{}."
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
    "Be concise."
)

SYSTEM_PROMPT_MCQ = (
    "You are a precise mathematical reasoner."
    "Read the problem and the answer choices below, then select the single best answer. "
    "After obtaining an answer, independently check it for errors or contradictions."
    "Output ONLY the letter of your final chosen option inside \\boxed{}, e.g. \\boxed{C}."
    "Be concise."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(
            f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options)
        )
        return (
            "<instructions>" + SYSTEM_PROMPT_MCQ + "</instructions>",
            f"<problem>{question}\n\nOptions:\n{opts_text}</problem>",
        )

    return (
        "<instructions>" + SYSTEM_PROMPT_MATH + "</instructions>",
        f"<problem>{question}</problem>",
    )

# Quick prompt preview
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} system prompt ──")
    print(sys_p, "\n")
    print(f"── {label} user prompt ──")
    print(usr_p[:500], "...\n")

── MCQ system prompt ──
<instructions>You are a precise mathematical reasoner.Read the problem and the answer choices below, then select the single best answer. After obtaining an answer, independently check it for errors or contradictions.Output ONLY the letter of your final chosen option inside \boxed{}, e.g. \boxed{C}.Be concise.</instructions> 

── MCQ user prompt ──
<problem>Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().

Options:
A. Unchanged
B. Increased by ten percent
C. Reduced by one percent
D. Increased by one percent
E. Decreased by ten percent
F. Halved
G. Unable to determine
H. Doubled
I. Decreased by five percent
J. Expanded tenfold</problem> ...

── Free-form system prompt ──
<instructions>You are a precise mathematical reasoner. Solve the following problem rigorously.After obtaining an answer, independently check it for errors or contradictions. Return only the corrected final solution inside \boxed{}.If the p

## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [5]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token

# llm = LLM(
#     model=MODEL_ID,
#     quantization="bitsandbytes",
#     load_format="bitsandbytes",
#     enable_prefix_caching=False,
#     gpu_memory_utilization=0.50,
#     max_model_len=16384,
#     trust_remote_code=True,
#     max_num_seqs=256,
#     max_num_batched_tokens=32768,
# )


# sampling_params = SamplingParams(
#     max_tokens=MAX_TOKENS,
#     temperature=0.6,
#     top_p=0.95,
#     top_k=20,
#     min_p=0.0,
#     presence_penalty=0.0,
#     repetition_penalty=1.0,
# )

# print("\n === Model loaded. ===\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 05-30 20:57:50 [utils.py:253] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 8192, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'max_num_batched_tokens': 8192, 'max_num_seqs': 64, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 05-30 20:57:51 [model.py:631] Resolved architecture: Qwen3ForCausalLM
INFO 05-30 20:57:51 [model.py:1745] Using max model len 8192
INFO 05-30 20:57:55 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 05-30 20:57:57 [system_utils.py:103] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 05-30 20:59:02 [llm.py:352] Supported tasks: ['generate']

=== Model loaded. ===



In [6]:
# Clear any old model from GPU before loading
import gc
import torch

try:
    del llm
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.85,
    max_model_len=32768,
    trust_remote_code=True,
    max_num_seqs=16,
    max_num_batched_tokens=8192,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("\n=== Model loaded for 16k run. ===\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 05-30 23:16:47 [utils.py:253] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 32768, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'max_num_batched_tokens': 8192, 'max_num_seqs': 16, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 05-30 23:16:48 [model.py:631] Resolved architecture: Qwen3ForCausalLM
INFO 05-30 23:16:48 [model.py:1745] Using max model len 32768
INFO 05-30 23:16:51 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 05-30 23:16:53 [system_utils.py:103] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 05-30 23:17:55 [llm.py:352] Supported tasks: ['generate']

=== Model loaded for 16k run. ===



In [7]:
prompts = []

for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

print(f"Built {len(prompts)} prompts.")
print("Example prompt preview:")
print(prompts[0][:1000])

Built 943 prompts.
Example prompt preview:
<|im_start|>system
<instructions>You are a precise mathematical reasoner. Solve the following problem rigorously.After obtaining an answer, independently check it for errors or contradictions. Return only the corrected final solution inside \boxed{}.If the problem has multiple sub-answers, separate them by commas inside a single \boxed{}, e.g. \boxed{3, 7}.Be concise.</instructions><|im_end|>
<|im_start|>user
<problem>Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]
b) $4 \cdot 3-2+2 \cdot 3=$ [ANS]</problem><|im_end|>
<|im_start|>assistant
<think>



## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [8]:
from datetime import datetime
from zoneinfo import ZoneInfo

# Build prompts for all entries
prompts = []

for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate all responses in one vLLM batched pass
start_time = datetime.now(ZoneInfo("America/Los_Angeles"))
print(f"Generation started at: {start_time.strftime('%Y-%m-%d %I:%M:%S %p %Z')}")

print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Sanity check: make sure every data item has one response
assert len(responses) == len(data), f"Expected {len(data)} responses, got {len(responses)}"

# Save raw responses immediately
Path("results").mkdir(exist_ok=True)

with open(OUTPUT_PATH, "w") as f:
    for idx, (item, response) in enumerate(zip(data, responses)):
        row = {
            "id": item.get("id"),
            "index": idx,
            "is_mcq": bool(item.get("options")),
            "question": item["question"],
            "gold": item.get("answer"),
            "response": response,
        }
        f.write(json.dumps(row) + "\n")
        f.flush()

end_time = datetime.now(ZoneInfo("America/Los_Angeles"))
elapsed = end_time - start_time

print(f"Saved raw responses to {OUTPUT_PATH}")
print(f"Generation ended at: {end_time.strftime('%Y-%m-%d %I:%M:%S %p %Z')}")
print(f"Total elapsed time: {elapsed}")

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generation started at: 2026-05-30 04:18:28 PM PDT
Generating responses for 943 questions...


Adding requests:   0%|          | 0/943 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/943 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Saved raw responses to results/verification-pe-colab-16k.jsonl
Generation ended at: 2026-05-30 07:16:34 PM PDT
Total elapsed time: 2:58:05.737176

── Response 0 (id=0) ──
Okay, let's tackle these two problems one by one. First, part a: [13 - (11 - 11)] - [8 - (5 - 6)]. 

I need to remember the order of operations, which is parentheses first, then exponents (but there are none here), then multiplication and division, then addition and subtraction. But here, the main thing is the innermost parentheses first.

Starting with the first bracket: [13 - (11 - 11)]. Let's c ...

── Response 1 (id=1) ──
Okay, let's try to figure out this problem. So the question says: "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ()". Hmm, first I need to understand what exactly is being asked here.

Wait, the problem mentions "weights corresponding to the sign values". Hmm, maybe "sign values" here refers to something like positive and negative values? O

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!